# Import and Shared functions

In [ ]:
import os, sys, re, ast
from glob import glob
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

AXES   = ["M1", "M2", "M3"]        # Base-frame motor axes -- shared by every θ/ω field in both KFLOG and PIDLOG
TOPO   = ["Az", "Alt", "Roll"]     # Topocentric frame (α_*) axes
EQU    = ["RA", "Dec", "PA"]       # Equatorial frame (Δ_*) axes -- where issue #88 shows up
ARCSEC = 3600                      # deg -> arcsec, for readability at the scale this investigation cares about

def wrap_deg(x):
    """Wrap a degree value/difference into (-180, 180]. Every 'error' plot below is pv - sp,
    which needs this: without it, sp=359.9/pv=0.1 (e.g. Az crossing 0/360, or an RA/PA axis
    wrapping) gives a spurious -359.8 instead of the true +0.2."""
    return (x + 180) % 360 - 180

def parse_val(v):
    v = v.strip()
    try:
        return float(v)
    except ValueError:
        return v

def parse_payload_line(line, tag):
    """Split 'TIMESTAMP INFO <TAG> {dict}' and literal_eval the trailing dict."""
    if f" {tag} " not in line:
        return None
    ts, _, body = line.partition(f" {tag} ")
    ts = ts.split(" INFO")[0].strip()
    try:
        payload = ast.literal_eval(body.strip())
    except (ValueError, SyntaxError):
        return None
    rec = {"timestamp": ts}
    for key, val in payload.items():
        if isinstance(val, list):
            for i, v in enumerate(val):
                # None (e.g. PECLOG's guide field on a cycle where only one axis updated) must
                # not be stringified to "None" first -- float("None") raises, so parse_val would
                # otherwise leave the literal string "None" sitting in a numeric column.
                rec[f"{key}_{i+1}"] = float("nan") if v is None else parse_val(str(v))
        else:
            rec[key] = float("nan") if val is None else parse_val(str(val))
    return rec

def resolve_log_files(pattern):
    """
    Expand a glob pattern (or accept an explicit list) into the set of rotated driver
    logs to load. A multi-hour capture routinely spans several rotated files regardless
    of the configured per-file size cap, so this is the normal path, not a special case.
    Only matches 'alpaca.log' and 'alpaca.log.<N>' -- excludes renamed/archived variants
    (e.g. 'alpaca.mark_*.log') and stray ':Zone.Identifier' sidecar files.
    """
    if isinstance(pattern, (list, tuple)):
        return list(pattern)
    candidates = glob(pattern)
    return [p for p in candidates if re.search(r"alpaca\.log(\.\d+)?$", os.path.basename(p))]

def load_kf_pid(log_paths):
    """
    Parse KFLOG and PIDLOG lines from one or more rotated driver logs
    (Config.log_position = true) into two DataFrames. Accepts a single path, a glob
    pattern (e.g. '../logs/alpaca.log*'), or an explicit list of paths -- all rows are
    concatenated and re-sorted by timestamp, so file order/naming doesn't matter.
    """
    paths = resolve_log_files(log_paths) if isinstance(log_paths, str) else list(log_paths)
    if not paths:
        raise FileNotFoundError(f"No log files matched: {log_paths!r}")
    kf_rows, pid_rows = [], []
    for log_path in paths:
        if not os.path.exists(log_path):
            raise FileNotFoundError(f"log_path does not exist: {log_path!r}")
        # encoding='utf-8' is required: the driver writes θ/ω/Δ/α as UTF-8 (log.py's
        # RotatingFileHandler is pinned to utf-8), but open() without an explicit encoding
        # falls back to the OS default -- cp1252 on Windows -- which silently mangles those
        # keys instead of raising, so columns like "θ_sp_1" quietly never get created.
        with open(log_path, encoding="utf-8") as f:
            for line in f:
                if " KFLOG " in line:
                    rec = parse_payload_line(line, "KFLOG")
                    if rec: kf_rows.append(rec)
                elif " PIDLOG " in line:
                    rec = parse_payload_line(line, "PIDLOG")
                    if rec: pid_rows.append(rec)

    if not kf_rows and not pid_rows:
        raise ValueError(f"No KFLOG/PIDLOG lines found in {paths!r} -- was Config.log_position true during this session?")

    def finalize(rows):
        df = pd.DataFrame(rows)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.drop_duplicates(subset="timestamp").sort_values("timestamp").reset_index(drop=True)
        df["t_sec"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
        df["gap_sec"] = df["timestamp"].diff().dt.total_seconds()
        return df

    kf_df  = finalize(kf_rows)  if kf_rows  else pd.DataFrame()
    pid_df = finalize(pid_rows) if pid_rows else pd.DataFrame()
    return kf_df, pid_df


# Load data

In [ ]:
# ── Choose log path (last set log_path is what is used) ────────────────────────
# Single file:            '../logs/alpaca.log'
# All rotated files:      '../logs/alpaca.log*'   (glob -- handles a multi-hour run spanning several files)
# Explicit file list:     ['../logs/alpaca.log.2', '../logs/alpaca.log.1', '../logs/alpaca.log']
log_path = ['../logs/logs/alpaca.soak_nopec_Beta4.3_08_27a.log', '../logs/logs/alpaca.soak_nopec_Beta4.3_08_27b.log']
log_path = ['../logs/logs/alpaca.soak_nopec_Beta4.3_08_28a.log']
log_path = ['../logs/logs/alpaca.soak_nopec_Beta4.3_08_28k1.log','../logs/logs/alpaca.soak_nopec_Beta4.3_08_28k2.log','../logs/logs/alpaca.soak_nopec_Beta4.3_08_28k3.log','../logs/logs/alpaca.soak_nopec_Beta4.3_08_28k4.log']


resolved_files = resolve_log_files(log_path) if isinstance(log_path, str) else list(log_path)
print(f"Loading {len(resolved_files)} file(s):")
for p in resolved_files:
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print()

kf_df, pid_df = load_kf_pid(log_path)
print(f"KF  samples: {len(kf_df)}"  + (f"  ({kf_df.t_sec.iloc[-1]/60:.1f} min span)"  if len(kf_df)  else ""))
print(f"PID samples: {len(pid_df)}" + (f"  ({pid_df.t_sec.iloc[-1]/60:.1f} min span)" if len(pid_df) else ""))
if len(kf_df):
    print(f"KF  time range: {kf_df.timestamp.iloc[0]}  ->  {kf_df.timestamp.iloc[-1]}")
if len(pid_df):
    print(f"PID time range: {pid_df.timestamp.iloc[0]}  ->  {pid_df.timestamp.iloc[-1]}")
print()
print("KF columns:", list(kf_df.columns))
print("PID columns:", list(pid_df.columns))


# Exposure subgrouping (from PECLOG)

`PECLOG` (parsed the same way as `analyse_pec.ipynb`) is written once per guide-sync cycle
(`control.py:_pec_log()`, called from `process_guide_sync()`) -- it marks a sync-guide event,
**not** one-per-exposure. If consecutive PECLOG entries are further apart than `EXPOSURE_SEC`,
that gap is several back-to-back exposures, not one long idle gap -- e.g. two PECLOGs 140s
apart at the 30s default means one sync-guide followed by ~4 exposures before the next sync.
This tags every PID/KF tick with which of those back-to-back exposures it falls in (and where
within that exposure), so anomalies can be grouped/filtered by exposure -- e.g. "which
sub-exposures had a flagged RA/PA event during them" -- rather than only viewed against the
whole session. Optional: this capture only has PECLOG entries if guiding was active during it.

In [ ]:
EXPOSURE_SEC = 30   # configurable -- assumed exposure duration; exposures are assumed to run
                     # back-to-back starting shortly after each PECLOG (see markdown above)

def load_peclog(log_paths):
    """
    Parse PECLOG lines using the same dict-payload convention as KFLOG/PIDLOG (see
    parse_payload_line above) -- control.py's _pec_log() emits one dict per guide-sync
    cycle. Returns an empty DataFrame (not an error) when none are found -- guiding/exposing
    isn't active in every capture, so this is optional/supplementary.
    """
    paths = resolve_log_files(log_paths) if isinstance(log_paths, str) else list(log_paths)
    rows = []
    for log_path in paths:
        with open(log_path, encoding="utf-8") as f:
            for line in f:
                if " PECLOG " not in line:
                    continue
                rec = parse_payload_line(line, "PECLOG")
                if rec: rows.append(rec)
    df = pd.DataFrame(rows)
    if len(df):
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.drop_duplicates(subset="timestamp").sort_values("timestamp").reset_index(drop=True)
    return df

peclog_df = load_peclog(log_path)

if len(peclog_df) and len(pid_df):
    origin = kf_df.timestamp.iloc[0] if len(kf_df) else pid_df.timestamp.iloc[0]
    peclog_df["t_sec"] = (peclog_df.timestamp - origin).dt.total_seconds()

    guide_starts = peclog_df.t_sec.to_numpy()
    t = pid_df.t_sec.to_numpy()
    guide_block = np.searchsorted(guide_starts, t, side="right") - 1   # which PECLOG-to-PECLOG interval this tick is in
    has_block = guide_block >= 0
    elapsed = np.full(len(pid_df), np.nan)                             # seconds since the most recent PECLOG
    elapsed[has_block] = t[has_block] - guide_starts[guide_block[has_block]]

    # Consecutive EXPOSURE_SEC windows repeat back-to-back within each guide interval, not
    # just a single window right after the PECLOG -- see markdown above.
    exposure_num = np.full(len(pid_df), -1, dtype=int)
    exposure_num[has_block] = np.floor(elapsed[has_block] / EXPOSURE_SEC).astype(int)
    exposure_sec = np.where(has_block, elapsed - exposure_num * EXPOSURE_SEC, np.nan)  # position within that specific exposure

    pid_df["guide_block"]  = guide_block                                          # -1 = before the first PECLOG (no exposure info yet)
    pid_df["exposure_num"] = exposure_num                                         # which back-to-back exposure within this guide interval (0-indexed)
    pid_df["exposure_sec"] = exposure_sec                                         # elapsed seconds within that specific exposure
    pid_df["exposure_id"]  = np.where(has_block, guide_block * 100000 + exposure_num, -1)  # unique id across the whole run, for groupby

    n_exposures = pid_df.loc[pid_df.exposure_id >= 0, "exposure_id"].nunique()
    print(f"PECLOG entries: {len(peclog_df)}")
    print(f"Tagged {has_block.sum()} of {len(pid_df)} PID ticks ({has_block.mean()*100:.1f}%) across "
          f"{n_exposures} inferred {EXPOSURE_SEC:.0f}s exposure(s) spanning {len(peclog_df)} guide interval(s).")
else:
    pid_df["guide_block"]  = -1
    pid_df["exposure_num"] = -1
    pid_df["exposure_id"]  = -1
    pid_df["exposure_sec"] = np.nan
    print(f"No PECLOG entries in this capture -- exposure subgrouping unavailable for this run "
          f"(default exposure length is {EXPOSURE_SEC}s; re-run once a capture with active guiding is loaded).")


# Telemetry gaps (known 518-dropout artifact)

Per `docs/control.md`: the Polaris sometimes stops sending 518 telemetry for a few seconds,
during which `θ_pv` freezes while `θ_sp` keeps advancing -- producing a large but spurious
error spike once telemetry resumes, that looks like a control anomaly but isn't one. Flag
any gap much longer than the normal ~0.1-0.2s KFLOG cadence, before any plotting below, so
a brief total dropout can be masked out of every chart instead of blowing out its y-axis and
hiding the real, subtler variation those charts exist to show.

In [ ]:
GAP_THRESHOLD_SEC = 1.0   # normal cadence is ~0.1-0.2s; the known artifact runs a few seconds

if len(kf_df):
    gaps_df = kf_df[kf_df.gap_sec > GAP_THRESHOLD_SEC][["timestamp", "t_sec", "gap_sec"]].reset_index(drop=True)
    print(f"Flagged {len(gaps_df)} telemetry gap(s) > {GAP_THRESHOLD_SEC}s, "
          f"totalling {gaps_df.gap_sec.sum():.1f}s of lost telemetry")
else:
    gaps_df = pd.DataFrame(columns=["timestamp", "t_sec", "gap_sec"])
    print("No KF data loaded.")
gaps_df


# Gap masking (applies to every plot below)

Tags every KF/PID tick within a window around a flagged gap as `near_gap`. Every chart below
plots `series.mask(near_gap)` rather than the raw series -- pandas turns those ticks to NaN,
so plotly draws a visible break in the line instead of connecting through it or including it
in the y-axis autorange. The window covers the gap's *entire actual duration* (from
`gaps_df.gap_sec`, not a fixed constant -- some dropouts run over 100s, not a few seconds)
plus a small pad before it and a longer tail after it, since the KF/PID visibly takes tens of
seconds to resettle once telemetry resumes -- calibrated against a real run, widen/narrow
`GAP_EXCLUDE_BEFORE`/`GAP_EXCLUDE_AFTER` as needed.

In [ ]:
GAP_EXCLUDE_BEFORE = 2.0   # extra seconds before the last good sample to mask, as a safety pad
GAP_EXCLUDE_AFTER  = 30.0  # seconds after a gap to mask -- KF/PID visibly takes this long to resettle

def mask_near_gap(df, gaps_df, before_pad=GAP_EXCLUDE_BEFORE, after=GAP_EXCLUDE_AFTER):
    near = pd.Series(False, index=df.index)
    for _, g in gaps_df.iterrows():
        start = g.t_sec - g.gap_sec - before_pad   # back to the actual last good sample, not a fixed offset --
        end   = g.t_sec + after                     # a dropout can run well over 100s, not just a few
        near |= (df.t_sec >= start) & (df.t_sec <= end)
    return near

kf_df["near_gap"]  = mask_near_gap(kf_df, gaps_df)  if len(kf_df)  else pd.Series(dtype=bool)
pid_df["near_gap"] = mask_near_gap(pid_df, gaps_df) if len(pid_df) else pd.Series(dtype=bool)

if len(kf_df):
    print(f"KF  ticks masked as gap-affected: {kf_df.near_gap.sum()} / {len(kf_df)} ({kf_df.near_gap.mean()*100:.1f}%)")
if len(pid_df):
    print(f"PID ticks masked as gap-affected: {pid_df.near_gap.sum()} / {len(pid_df)} ({pid_df.near_gap.mean()*100:.1f}%)")


# Startup masking (applies to every plot below, same as gap masking)

A fresh driver start has a real, one-off transient, in *both* streams: `theta_ref` isn't established until the first control cycle runs, so the earliest KFLOG tick(s) can carry raw absolute angles (seen directly in a real capture: M1/M2 `theta_meas`/`theta_state` over 100 degrees on the very first sample) instead of the near-zero reference-relative residual every later tick has, and PIDLOG's `Delta_sp` jumps by the full initial setpoint-establishment step in the first ~0.2s. Both settle within `STARTUP_EXCLUDE_SEC`. Masked the same way as `near_gap` (as `near_startup`), and combined into one `exclude` column used everywhere below -- so every plot/metric excludes it consistently instead of each cell reinventing its own threshold (this is exactly what happened before: the RA/PA anomaly hunt further down had its own local startup exclusion, but the KF plots/variation-reduction report above it didn't).

In [ ]:
STARTUP_EXCLUDE_SEC = 30.0   # seconds from the start of the run to mask as startup transient

kf_df["near_startup"]  = (kf_df.t_sec  < STARTUP_EXCLUDE_SEC) if len(kf_df)  else pd.Series(dtype=bool)
pid_df["near_startup"] = (pid_df.t_sec < STARTUP_EXCLUDE_SEC) if len(pid_df) else pd.Series(dtype=bool)

kf_df["exclude"]  = (kf_df.near_gap  | kf_df.near_startup)  if len(kf_df)  else pd.Series(dtype=bool)
pid_df["exclude"] = (pid_df.near_gap | pid_df.near_startup) if len(pid_df) else pd.Series(dtype=bool)

if len(kf_df):
    print(f"KF  ticks masked as startup transient: {kf_df.near_startup.sum()} / {len(kf_df)} "
          f"({kf_df.near_startup.mean()*100:.1f}%)")
if len(pid_df):
    print(f"PID ticks masked as startup transient: {pid_df.near_startup.sum()} / {len(pid_df)} "
          f"({pid_df.near_startup.mean()*100:.1f}%)")


# KF: Measured vs Filtered, position already reference-relative, velocity vs ω_ref

`θ_meas`/`θ_state` already have `theta_ref` subtracted by the driver whenever tracking is
enabled (`control.py`'s `observe()` passes `theta_ref=self._pid.theta_ref` into
`wrap_angle_residual()`) -- plotting them directly, with no further subtraction, is correct.
`ω_meas`/`ω_state` are different: KFLOG logs those completely raw
(`omega_meas.flatten().tolist()`, never passed through `wrap_angle_residual`), so comparing
them to `ω_ref` (also a KFLOG field) has to happen here in the notebook, not in the driver.

In [ ]:
fig = make_subplots(rows=3, cols=2, shared_xaxes=True,
    subplot_titles=sum([[f"{ax} -- θ_meas vs θ_state (arcsec, already ref-relative)", f"{ax} -- (ω_meas - ω_ref) vs (ω_state - ω_ref) (arcsec/s)"] for ax in AXES], []),
    vertical_spacing=0.06)
exclude = kf_df.exclude
for i, ax in enumerate(AXES):
    row = i + 1
    meas_err  = (kf_df[f"θ_meas_{row}"]  * ARCSEC).mask(exclude)
    state_err = (kf_df[f"θ_state_{row}"] * ARCSEC).mask(exclude)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=meas_err, name=f"{ax} θ_meas",
        line=dict(color="royalblue", width=1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=state_err, name=f"{ax} θ_state",
        line=dict(color="white", width=1)), row=row, col=1)
    meas_verr  = ((kf_df[f"ω_meas_{row}"]  - kf_df[f"ω_ref_{row}"]) * ARCSEC).mask(exclude)
    state_verr = ((kf_df[f"ω_state_{row}"] - kf_df[f"ω_ref_{row}"]) * ARCSEC).mask(exclude)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=meas_verr, name=f"{ax} ω_meas-ω_ref",
        line=dict(color="royalblue", width=1)), row=row, col=2)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=state_verr, name=f"{ax} ω_state-ω_ref",
        line=dict(color="white", width=1)), row=row, col=2)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=2)
fig.update_layout(height=900, width=1500, template="plotly_dark", hovermode="x unified",
    title="Kalman Filter -- Measured vs Filtered, relative to reference", legend=dict(groupclick="toggleitem"))
fig.show()


# KF: variation reduced vs the raw input signal

Reports `std(θ_state)` against `std(θ_meas)` per axis (and the equivalent against `ω_ref`
for `ω`, see markdown above) as a direct "how much noise did the filter remove" number --
gap-affected ticks excluded, since a few huge dropout-recovery spikes would otherwise
dominate a std() and make the filter look far more/less effective than it actually is on
ordinary ticks.

A large std here doesn't necessarily mean noise, though -- it's just as consistent with a
real, systematic bias or drift (constant offset, or a steady trend over the session) sitting
underneath. Since std alone can't tell those apart, this also reports mean and a linear
time-trend per axis: a mean far from zero with a *small* trend looks like a persistent bias;
a small mean with a large trend looks like something drifting away over the session (e.g. an
uncorrected/mis-signed periodic or secular term); large std with both small is closer to
genuine noise.

In [ ]:
clean_kf = kf_df[~kf_df.exclude] if len(kf_df) else kf_df

SANE_STD_ARCSEC = 30.0   # flag axes whose std is this large -- worth characterizing further below

def report_reduction(label, unit, meas, state):
    raw_std  = meas.std()
    filt_std = state.std()
    pct = (1 - filt_std / raw_std) * 100 if raw_std > 0 else float("nan")
    word = "reduction" if pct >= 0 else "increase"
    flag = f"  <-- std > {SANE_STD_ARCSEC:.0f}, see characterization below" if raw_std > SANE_STD_ARCSEC else ""
    print(f"  {label}: raw std={raw_std:9.3f}{unit} -> filtered std={filt_std:9.3f}{unit}  ({pct:+.1f}% {word}){flag}")

def characterize(label, series, t_sec):
    slope = np.polyfit(t_sec, series, 1)[0] * 3600  # arcsec/hr
    half = t_sec.min() + (t_sec.max() - t_sec.min()) / 2
    first_half_mean  = series[t_sec < half].mean()
    second_half_mean = series[t_sec >= half].mean()
    print(f"    {label}: mean={series.mean():+9.2f}\"  trend={slope:+9.2f}\"/hr  "
          f"1st-half mean={first_half_mean:+9.2f}\"  2nd-half mean={second_half_mean:+9.2f}\"")

if len(clean_kf):
    print("Position (θ, arcsec):")
    large_axes = []
    for i, ax in enumerate(AXES):
        meas  = clean_kf[f"θ_meas_{i+1}"]  * ARCSEC
        state = clean_kf[f"θ_state_{i+1}"] * ARCSEC
        report_reduction(ax, '"', meas, state)
        if meas.std() > SANE_STD_ARCSEC:
            large_axes.append((ax, meas))

    print("\nVelocity (ω - ω_ref, arcsec/s):")
    for i, ax in enumerate(AXES):
        meas_verr  = (clean_kf[f"ω_meas_{i+1}"]  - clean_kf[f"ω_ref_{i+1}"]) * ARCSEC
        state_verr = (clean_kf[f"ω_state_{i+1}"] - clean_kf[f"ω_ref_{i+1}"]) * ARCSEC
        report_reduction(ax, '"/s', meas_verr, state_verr)

    if large_axes:
        print(f"\nCharacterizing axes with std > {SANE_STD_ARCSEC:.0f}\" (bias vs. drift vs. noise):")
        for ax, meas in large_axes:
            characterize(ax, meas, clean_kf.t_sec)
else:
    print("No clean KF data to compute a reduction metric from.")

# KF: Gain per axis (position + velocity)

`K_gain` is the diagonal of the Kalman gain matrix: indices 1-3 are the position gains
(θ1-3), 4-6 the velocity gains (ω1-3). A gain spike means the filter suddenly started
trusting a raw measurement much more than usual -- worth cross-checking against any
θ_meas/θ_state divergence at the same tick.

In [ ]:
fig = go.Figure()
labels = [f"{ax} pos" for ax in AXES] + [f"{ax} vel" for ax in AXES]
colors = ["royalblue", "orange", "mediumseagreen", "royalblue", "orange", "mediumseagreen"]
dashes = ["solid", "solid", "solid", "dash", "dash", "dash"]
for i, (label, color, dash) in enumerate(zip(labels, colors, dashes)):
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=kf_df[f"K_gain_{i+1}"].mask(kf_df.exclude), name=label,
        line=dict(color=color, width=1, dash=dash)))
fig.update_layout(height=500, width=1300, template="plotly_dark", hovermode="x unified",
    title="Kalman Gain per axis", xaxis_title="Time (s)", yaxis_title="Gain",
    legend=dict(groupclick="toggleitem"))
fig.show()


# PID: Position tracking error (θ, Base-frame motor angles)

Plotted as `θ_pv - θ_sp` (arcsec, wrap-safe -- see `wrap_deg`) rather than the two raw lines
overlaid -- M1-M3 drift together as the mount tracks, so an SP-vs-PV overlay is two
near-identical slowly-moving lines with the actual tracking error invisible at that scale.
This is the error directly.

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- θ_pv - θ_sp (arcsec)" for ax in AXES], vertical_spacing=0.08)
for i, ax in enumerate(AXES):
    row = i + 1
    error = (wrap_deg(pid_df[f"θ_pv_{row}"] - pid_df[f"θ_sp_{row}"]) * ARCSEC).mask(pid_df.exclude)
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=error, name=f"{ax} error",
        line=dict(color="white", width=1)), row=row, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=800, width=1300, template="plotly_dark", hovermode="x unified",
    title="PID Position Tracking Error", legend=dict(groupclick="toggleitem"))
fig.show()


# PID: Velocity breakdown (Kp / Ki / Kd / FF / Output)

Matches the Alpaca Pilot PID Tuning page convention: Cyan = Output (ω_op, what actually
drove the motor), Magenta = Kp, Olive = Ki, Orange = Kd, Green = FF (currently ω_ff − ω_pec
combined -- PEC is not yet broken out as its own field in the payload).

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- velocity components (arcsec/s)" for ax in AXES], vertical_spacing=0.08)
components = [("ω_ff", "green"), ("ω_kp", "magenta"), ("ω_ki", "olive"), ("ω_kd", "orange"), ("ω_op", "cyan")]
for i, ax in enumerate(AXES):
    row = i + 1
    for key, color in components:
        fig.add_trace(go.Scatter(x=pid_df.t_sec, y=(pid_df[f"{key}_{row}"] * ARCSEC).mask(pid_df.exclude), name=f"{ax} {key}",
            line=dict(color=color, width=1)), row=row, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=900, width=1300, template="plotly_dark", hovermode="x unified",
    title="PID Velocity Breakdown", legend=dict(groupclick="toggleitem"))
fig.show()


# PID: Equatorial & Topocentric tracking error (RA/Dec/PA, Az/Alt/Roll)

Issue #88's reported symptom (near-meridian RA/PA noise) shows up in the **equatorial**
frame (`Δ_*`), not the base motor frame (`θ_*`) plotted above -- M1/M2/M3 look clean even
during an affected period, so this view is the one that actually matters for that
investigation. Both columns plot `pv - sp` (error, wrap-safe -- see `wrap_deg`) rather than
an SP/PV overlay. RA/Dec need this because they barely move during single-target tracking, so
the raw lines would overlap and hide the error -- but Az/Alt/Roll need it too, for a different
reason: they *do* move a lot over a session (tens of degrees), so the actual tracking error
(arcsec-scale) would be completely invisible against that range on a raw overlay, and Az
crossing 0°/360° would otherwise show as a spurious ~360° spike without the wrap-safe diff.
`α_pv_1` = Az, marked against meridian transit (Az≈178-180°) with a dotted line for reference.

In [ ]:
fig = make_subplots(rows=3, cols=2, shared_xaxes=True,
    subplot_titles=sum([[f"{ax} -- Δ_pv - Δ_sp (arcsec)", f"{TOPO[i]} -- α_pv - α_sp (arcsec)"] for i, ax in enumerate(EQU)], []),
    vertical_spacing=0.06)
exclude = pid_df.exclude
for i in range(3):
    row = i + 1
    equ_error  = (wrap_deg(pid_df[f"Δ_pv_{row}"] - pid_df[f"Δ_sp_{row}"]) * ARCSEC).mask(exclude)
    topo_error = (wrap_deg(pid_df[f"α_pv_{row}"] - pid_df[f"α_sp_{row}"]) * ARCSEC).mask(exclude)
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=equ_error, name=f"{EQU[i]} error",
        line=dict(color="white", width=1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=topo_error, name=f"{TOPO[i]} error",
        line=dict(color="white", width=1)), row=row, col=2)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=2)
fig.update_layout(height=900, width=1500, template="plotly_dark", hovermode="x unified",
    title="PID Equatorial (left) & Topocentric (right) Tracking Error",
    legend=dict(groupclick="toggleitem"))
fig.show()
# Az range / meridian-proximity is checked separately in the RA/PA anomaly-hunt section below,
# since absolute Az isn't on this chart anymore now that the right column is error, not SP/PV.


# Combined KF+PID timeline

Merges the two streams on nearest timestamp (they tick independently -- KF on every 518
message, PID on every control step -- so this aligns them within a short tolerance rather
than assuming a shared clock). Used by the "zoom into context" cells below.

In [ ]:
TOLERANCE = pd.Timedelta("100ms")

merged = pd.DataFrame()
if len(kf_df) and len(pid_df):
    merged = pd.merge_asof(
        pid_df.sort_values("timestamp"), kf_df.sort_values("timestamp"),
        on="timestamp", direction="nearest", tolerance=TOLERANCE,
        suffixes=("_pid", "_kf"),
    )
    merged["t_sec"] = (merged["timestamp"] - merged["timestamp"].iloc[0]).dt.total_seconds()
print(f"Merged samples: {len(merged)}")
merged.head()


# Anomaly hunt: sudden PID reactions

Flags *events* -- runs of consecutive control ticks where `|ω_kp|` on any axis sits far
above its local baseline -- a direct proxy for "the PID suddenly thought it was a long
way off target and kicked the motor hard", the kind of transient that could put a bend
in a star trail without ever crossing the log's `WARNING` lag thresholds. Uses a rolling
median + MAD (robust to the spikes themselves, unlike mean/stdev) for the per-tick
threshold. Flagging combines two rules so both shapes of disturbance get caught without
drowning in one-off noise: a single tick clearing the high `N_MAD_SPIKE` bar is an event
on its own (a lone sharp kick), while a lower `N_MAD_RUN` bar only counts with
`MIN_RUN`+ consecutive ticks (a slower multi-second "hump" that never spikes hard on any
one tick). Consecutive flagged ticks on the same axis within `CLUSTER_GAP_S` of each
other are merged into a single event (start/end/peak) rather than listed per-tick.

`MANUAL_EVENTS` below is an escape hatch for anything spotted by eye (e.g. in the event
explorer) that the automatic thresholds don't clear -- add `(axis, t_sec)` and it's
folded into the same table/explorer, tagged `manual=True`, with no detection logic
applied.

In [ ]:
WINDOW = 51          # ticks (~10s at 200ms) for the rolling baseline
N_MAD_SPIKE = 8.0    # a single tick this far above baseline is an event on its own
N_MAD_RUN = 5.0      # a lower bar, but only counts with >=MIN_RUN consecutive ticks
MIN_RUN = 2          # minimum consecutive flagged ticks (same axis) for the N_MAD_RUN bar to count
CLUSTER_GAP_S = 1.5  # merge flagged ticks on the same axis into one event if within this many
                     # seconds of each other, so one multi-tick disturbance is one row, not N

MANUAL_EVENTS = [
    ("M2", 1457.04),  # spotted by eye in the explorer around the M2 @ 1447.2s event -- a real
                       # ~2.4s sustained excursion, but its peak (~1.7 arcsec/s) never clears
                       # N_MAD_RUN against this stretch's low local baseline (~0.7), so automatic
                       # detection can't reach it without flooding the whole run with false positives
]

events = []
axis_stats = {}
for i, ax in enumerate(AXES):
    col = f"ω_kp_{i+1}"
    series = (pid_df[col] * ARCSEC).abs()
    med = series.rolling(WINDOW, center=True, min_periods=WINDOW//2).median()
    mad = (series - med).abs().rolling(WINDOW, center=True, min_periods=WINDOW//2).median()
    robust_std = 1.4826 * mad
    axis_stats[ax] = (col, med)

    candidate = (series > med + N_MAD_RUN * robust_std) & (robust_std > 1e-6) & (~pid_df.exclude)
    flagged = pid_df.loc[candidate, ["t_sec", "timestamp"]].copy()
    flagged["val"] = pid_df.loc[candidate, col] * ARCSEC
    flagged["is_spike"] = (series[candidate] > (med[candidate] + N_MAD_SPIKE * robust_std[candidate])).values
    if not len(flagged):
        continue
    group = (flagged.t_sec.diff().fillna(0) > CLUSTER_GAP_S).cumsum()
    for _, g in flagged.assign(group=group).groupby("group"):
        if len(g) < MIN_RUN and not g.is_spike.any():
            continue
        peak = g.loc[g.val.abs().idxmax()]
        events.append(dict(axis=ax, t_sec=peak.t_sec, timestamp=peak.timestamp,
                            t_start=g.t_sec.min(), t_end=g.t_sec.max(), n_ticks=len(g),
                            omega_kp_arcsec_s=peak.val, local_median_arcsec_s=med.loc[peak.name],
                            manual=False))

for ax, t_center in MANUAL_EVENTS:
    col, med = axis_stats[ax]
    nearest = (pid_df.t_sec - t_center).abs().idxmin()
    events.append(dict(axis=ax, t_sec=pid_df.t_sec[nearest], timestamp=pid_df.timestamp[nearest],
                        t_start=pid_df.t_sec[nearest], t_end=pid_df.t_sec[nearest], n_ticks=1,
                        omega_kp_arcsec_s=pid_df[col][nearest] * ARCSEC,
                        local_median_arcsec_s=med[nearest], manual=True))

anomalies_df = pd.DataFrame(events).sort_values("t_sec") if events else pd.DataFrame()
print(f"Flagged {len(anomalies_df)} anomalous event(s) across {len(AXES)} axes (telemetry-gap ticks excluded)")
anomalies_df.head(30)

In [ ]:
# Zoom into context (±5s) around the single largest flagged event, across every PID/KF component.
if len(anomalies_df):
    worst = anomalies_df.loc[anomalies_df.omega_kp_arcsec_s.idxmax()]
    print(f"Largest event: axis={worst.axis}  peak t={worst.t_sec:.2f}s  "
          f"span=[{worst.t_start:.2f}, {worst.t_end:.2f}]s ({worst.n_ticks} ticks)  |ω_kp|={worst.omega_kp_arcsec_s:.3f} arcsec/s")
    window = merged[(merged.t_sec > worst.t_sec - 5) & (merged.t_sec < worst.t_sec + 5)] if len(merged) else pd.DataFrame()
    display_cols = [c for c in window.columns if c.startswith(("θ_", "ω_", "t_sec"))]
    window[display_cols]
else:
    print("No events flagged at this threshold -- try lowering N_MAD or MIN_RUN.")


# Anomaly hunt: event explorer

Interactive companion to the flagged events above -- pick any one from the dropdown (or
step through with the slider) and see ±20s of position and velocity traces around it,
with a shaded band marking the event's actual start/end (it may span several ticks) and
a dashed marker at its peak tick (t=0). Time axis is relative to the selected event's
peak rather than absolute session time, so different events line up for direct
comparison regardless of when in the run they occurred.

Four panels, top to bottom:
1. **518 timing (s)** -- the raw receipt interval (time since the previous 518, yellow)
   alongside `measurement_lag_s` (orange, requires a capture with the
   `measurement_lag_s`/`θ_meas_raw`/`θ_ref_raw` KFLOG fields -- older captures won't have
   these and the panel will just be empty). Same units, one axis: a tall yellow spike is
   one slow receipt; a tall orange one is *accumulated* backlog, which can persist for a
   few ticks after receipt has already recovered.
2. **θ_meas_dev / θ_state_dev (arcsec)** -- what used to be plotted as `θ_meas`/`θ_state`,
   renamed to make explicit that these are deviations from a *backdated* theta_ref, not
   raw position (see the KF section above). Two more traces sit on this panel but start
   **hidden** -- `θ_meas_true`/`θ_state_true`, the absolute (non-backdated, degrees, own
   secondary axis) values, reconstructed from the new raw KFLOG fields. Toggle them on
   via the legend to sanity-check a deviation spike against the real underlying
   measurement, exactly the kind of check that caught the reference-staleness bug in the
   first place. They reset to hidden when you switch events.
3. **θ error (arcsec)** -- PID position error, as before.
4. **velocity components (arcsec/s)** -- ω_ff/kp/ki/kd/op, as before.

In [ ]:
EVENT_WINDOW_SEC = 20.0
components = [("ω_ff", "green"), ("ω_kp", "magenta"), ("ω_ki", "olive"), ("ω_kd", "orange"), ("ω_op", "cyan")]

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
    specs=[[{}], [{"secondary_y": True}], [{}], [{}]],
    subplot_titles=["518 timing: receipt interval vs measurement_lag_s (s)",
                     "θ_meas_dev / θ_state_dev (arcsec) -- θ_meas_true / θ_state_true hidden, own axis (deg)",
                     "θ error (arcsec)", "velocity components (arcsec/s)"],
    row_heights=[0.2, 0.3, 0.2, 0.3], vertical_spacing=0.06)

if len(anomalies_df):
    events = anomalies_df.reset_index(drop=True)

    def event_shapes(ev):
        """Dashed line at the peak tick (t=0) plus a shaded band over the event's actual
        start/end span, both expressed relative to the peak so they match the traces' x-axis."""
        return [
            dict(type="line", xref="x", yref="paper", x0=0, x1=0, y0=0, y1=1,
                 line=dict(color="red", width=1, dash="dash")),
            dict(type="rect", xref="x", yref="paper",
                 x0=ev.t_start - ev.t_sec, x1=ev.t_end - ev.t_sec, y0=0, y1=1,
                 fillcolor="red", opacity=0.08, line_width=0),
        ]

    trace_spans = []
    true_trace_indices = []   # (meas_true_idx, state_true_idx) per event, for the always-hide-on-nav fixup below
    for ev_i, ev in events.iterrows():
        ax_idx = AXES.index(ev.axis) + 1
        visible = (ev_i == 0)
        start = len(fig.data)

        kf_window = kf_df[(kf_df.t_sec > ev.t_sec - EVENT_WINDOW_SEC) & (kf_df.t_sec < ev.t_sec + EVENT_WINDOW_SEC)] if len(kf_df) else pd.DataFrame()
        t_rel_kf = kf_window.t_sec - ev.t_sec

        # Row 1: raw 518 receipt interval + accumulated measurement lag, both in seconds
        fig.add_trace(go.Scatter(x=t_rel_kf, y=kf_window.gap_sec, name="518 receipt", visible=visible,
            mode="markers+lines", marker=dict(size=4, color="yellow"), line=dict(color="yellow", width=1)), row=1, col=1)
        lag_col = kf_window["measurement_lag_s"] if "measurement_lag_s" in kf_window.columns else pd.Series(dtype=float)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=lag_col, name="measurement_lag_s", visible=visible,
            mode="markers+lines", marker=dict(size=4, color="orange"), line=dict(color="orange", width=1)), row=1, col=1)

        # Row 2: θ_meas_dev/θ_state_dev (primary axis) + hidden-by-default true absolute traces (secondary axis)
        meas_dev  = (kf_window[f"θ_meas_{ax_idx}"]  * ARCSEC).mask(kf_window.exclude)
        state_dev = (kf_window[f"θ_state_{ax_idx}"] * ARCSEC).mask(kf_window.exclude)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=meas_dev, name=f"{ev.axis} θ_meas_dev", visible=visible,
            line=dict(color="royalblue", width=1)), row=2, col=1, secondary_y=False)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=state_dev, name=f"{ev.axis} θ_state_dev", visible=visible,
            line=dict(color="white", width=1)), row=2, col=1, secondary_y=False)

        meas_raw_col = f"θ_meas_raw_{ax_idx}"
        ref_raw_col  = f"θ_ref_raw_{ax_idx}"
        omega_ref_col = f"ω_ref_{ax_idx}"
        have_raw = meas_raw_col in kf_window.columns and ref_raw_col in kf_window.columns and "measurement_lag_s" in kf_window.columns
        if have_raw:
            meas_true = kf_window[meas_raw_col].mask(kf_window.exclude)
            theta_ref_backdated = kf_window[ref_raw_col] - kf_window[omega_ref_col] * kf_window["measurement_lag_s"]
            state_true = (kf_window[f"θ_state_{ax_idx}"] + theta_ref_backdated).mask(kf_window.exclude)
        else:
            meas_true = pd.Series(dtype=float)
            state_true = pd.Series(dtype=float)
        # always False at creation -- these are opt-in via the legend, not part of the per-event
        # visibility toggle, and reset to hidden when the dropdown/slider switches events
        meas_true_idx = len(fig.data)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=meas_true, name=f"{ev.axis} θ_meas_true", visible=False,
            line=dict(color="deepskyblue", width=1, dash="dot")), row=2, col=1, secondary_y=True)
        state_true_idx = len(fig.data)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=state_true, name=f"{ev.axis} θ_state_true", visible=False,
            line=dict(color="lightgray", width=1, dash="dot")), row=2, col=1, secondary_y=True)
        true_trace_indices.append((meas_true_idx, state_true_idx))

        window = pid_df[(pid_df.t_sec > ev.t_sec - EVENT_WINDOW_SEC) & (pid_df.t_sec < ev.t_sec + EVENT_WINDOW_SEC)]
        t_rel = window.t_sec - ev.t_sec
        error = (wrap_deg(window[f"θ_pv_{ax_idx}"] - window[f"θ_sp_{ax_idx}"]) * ARCSEC).mask(window.exclude)
        fig.add_trace(go.Scatter(x=t_rel, y=error, name=f"{ev.axis} error", visible=visible,
            line=dict(color="white", width=1)), row=3, col=1)
        for key, color in components:
            fig.add_trace(go.Scatter(x=t_rel, y=(window[f"{key}_{ax_idx}"] * ARCSEC).mask(window.exclude),
                name=f"{ev.axis} {key}", visible=visible, line=dict(color=color, width=1)), row=4, col=1)
        trace_spans.append((start, len(fig.data) - start))

    n_total = len(fig.data)
    buttons = []
    for ev_i, ev in events.iterrows():
        start, count = trace_spans[ev_i]
        visible = [False] * n_total
        for j in range(start, start + count):
            visible[j] = True
        # θ_meas_true/θ_state_true always stay hidden on navigation, matching their
        # "opt-in via legend" behaviour at creation, regardless of where in the span they fall
        for idx in true_trace_indices[ev_i]:
            visible[idx] = False
        tag = " [manual]" if ev.get("manual") else ""
        buttons.append(dict(
            label=f"{ev.axis} @ {ev.t_sec:.1f}s ({ev.omega_kp_arcsec_s:.1f} arcsec/s, {ev.n_ticks} ticks/{ev.t_end-ev.t_start:.1f}s){tag}",
            method="update",
            args=[{"visible": visible},
                  {"title": f"Event: {ev.axis} @ t={ev.t_sec:.1f}s  (span {ev.t_end-ev.t_start:.2f}s, {ev.n_ticks} ticks){tag}",
                   "shapes": event_shapes(ev)}],
        ))

    slider_steps = []
    for ev_i, ev in events.iterrows():
        b = dict(buttons[ev_i])
        b["label"] = f"{ev.axis}@{ev.t_sec:.0f}s"
        slider_steps.append(b)

    fig.add_hline(y=0.2, row=1, col=1, line=dict(color="gray", width=1, dash="dot"))
    fig.update_yaxes(title_text="θ absolute (deg)", row=2, col=1, secondary_y=True)
    fig.update_xaxes(title_text="Time relative to event peak (s)", row=4, col=1)
    fig.update_layout(height=1100, width=1300, template="plotly_dark", hovermode="x unified",
        title=f"Event: {events.iloc[0].axis} @ t={events.iloc[0].t_sec:.1f}s  "
              f"(span {events.iloc[0].t_end-events.iloc[0].t_start:.2f}s, {events.iloc[0].n_ticks} ticks)",
        shapes=event_shapes(events.iloc[0]),
        legend=dict(groupclick="toggleitem"),
        updatemenus=[dict(buttons=buttons, direction="down", x=1.0, xanchor="right", y=1.1, yanchor="top",
                           showactive=True, bgcolor="#2B2B2B", bordercolor="#777777", borderwidth=1,
                           font=dict(color="#EEEEEE"))],
        sliders=[dict(active=0, x=0.0, len=0.88, pad=dict(t=60, b=10),
                       currentvalue=dict(prefix="Event (use ←/→ or drag to step): ", font=dict(color="#EEEEEE", size=12)),
                       font=dict(color="#EEEEEE", size=10), bgcolor="#2B2B2B", bordercolor="#777777",
                       activebgcolor="#555555", steps=slider_steps)])
    fig.show()
else:
    print("No events flagged -- nothing to explore.")

# Anomaly hunt: RA/PA mirrored oscillation (issue #88 signature)

Operationalizes the specific symptom documented for issue #88: near meridian transit
(Az≈178-180°), RA and PA tracking error have been seen oscillating as near-exact mirror
images of each other (~4-5s period, ±3-5") while Dec and all three motor axes stay quiet.

**Calibration note (from a real 2.5h run):** rolling corr(error_RA, error_PA) turned out to
be strongly negative almost everywhere in this dataset (median ≈ -0.96, true on 98.8% of
ticks) -- so anti-correlation by itself is *not* a useful discriminator, it looks like a
generic geometric property of this coordinate transform rather than something specific to
the reported anomaly. The threshold below leans on **amplitude** instead, with correlation
kept only as a loose sanity floor. Telemetry-gap ticks are excluded using the shared
`near_gap` column computed earlier.

The cell also reports the Az range actually covered by the loaded run -- if it never comes
near 180°, that's not a negative result for the hypothesis, it just means this run can't
test it (see `docs/control.md`'s suggested next step: bracket Az values either side of 178°).

**Startup exclusion:** the first `STARTUP_EXCLUDE_SEC` of a run are dropped before flagging. A fresh driver start has a real, large one-off setpoint-establishment jump (seen directly in a real no-PEC 1h capture: `Δ_sp` jumped ~10° in the first 0.2s), which swamps the rolling-amplitude filter and gets flagged as the 'largest event' every time -- it's not oscillation, just initial convergence.

In [ ]:
CORR_WINDOW       = 41    # ticks (~8s at 200ms) -- spans ~2 cycles of the reported ~4-5s oscillation
CORR_THRESHOLD    = -0.3  # loose sanity floor only -- see calibration note above, this barely discriminates
AMP_THRESHOLD_ARC = 10.0  # rolling std of error_RA (arcsec) -- calibrated against a real run, tune per-dataset
MERIDIAN_WINDOW   = 5.0   # degrees either side of Az=180 counted as "near meridian"

ra_pa = pid_df[["timestamp", "t_sec", "near_gap", "near_startup", "exposure_id"]].copy()
ra_pa["error_RA"] = (pid_df["Δ_pv_1"] - pid_df["Δ_sp_1"]) * ARCSEC
ra_pa["error_PA"] = (pid_df["Δ_pv_3"] - pid_df["Δ_sp_3"]) * ARCSEC
ra_pa["az"]       = pid_df["α_pv_1"]

ra_pa["roll_corr"] = ra_pa["error_RA"].rolling(CORR_WINDOW, center=True, min_periods=CORR_WINDOW).corr(ra_pa["error_PA"])
ra_pa["roll_amp"]  = ra_pa["error_RA"].rolling(CORR_WINDOW, center=True, min_periods=CORR_WINDOW).std()

print(f"Az range covered by this run: {ra_pa.az.min():.1f}° -- {ra_pa.az.max():.1f}°")
if (ra_pa.az - 180).abs().min() > MERIDIAN_WINDOW:
    print(f"  *** This run never comes within {MERIDIAN_WINDOW:.0f}° of Az=180 -- it CANNOT test the "
          f"near-meridian hypothesis. Any events below are informative on their own merits only. ***")
print(f"Rolling corr(error_RA, error_PA): median={ra_pa.roll_corr.median():.3f}, "
      f"frac < -0.5 = {(ra_pa.roll_corr < -0.5).mean()*100:.1f}% of ticks (context for the calibration note above)")

flagged = ra_pa[(ra_pa.roll_corr < CORR_THRESHOLD) & (ra_pa.roll_amp > AMP_THRESHOLD_ARC) & (~ra_pa.near_startup)].copy()
flagged["near_meridian"] = (flagged.az - 180).abs() <= MERIDIAN_WINDOW

clean_flagged = flagged[~flagged.near_gap]
n_startup_dropped = (ra_pa.roll_corr < CORR_THRESHOLD).__and__(ra_pa.roll_amp > AMP_THRESHOLD_ARC).__and__(ra_pa.near_startup).sum()
print(f"\nFlagged {len(flagged)} tick(s) ({len(flagged)/len(ra_pa)*100:.2f}% of run) with elevated, anti-correlated "
      f"RA/PA error ({len(flagged) - len(clean_flagged)} dropped as telemetry-gap artifacts, "
      f"{n_startup_dropped} dropped as first-{STARTUP_EXCLUDE_SEC:.0f}s startup transient).")
if len(clean_flagged):
    print(f"Of those, {clean_flagged.near_meridian.sum()}/{len(clean_flagged)} were within "
          f"±{MERIDIAN_WINDOW:.0f}° of Az=180° (meridian), and "
          f"{(clean_flagged.exposure_id >= 0).sum()}/{len(clean_flagged)} fell inside a tagged exposure window.")

    # Collapse consecutive flagged ticks into discrete events for a readable summary
    tick_dt = pid_df.t_sec.diff().median()
    is_new_event = clean_flagged.t_sec.diff().fillna(np.inf) > (tick_dt * 3)
    clean_flagged["event_id"] = is_new_event.cumsum()
    events = clean_flagged.groupby("event_id").agg(
        start_sec=("t_sec", "min"), end_sec=("t_sec", "max"),
        n_ticks=("t_sec", "size"), az_min=("az", "min"), az_max=("az", "max"),
        worst_corr=("roll_corr", "min"), peak_amp_arcsec=("roll_amp", "max"),
        exposure_id=("exposure_id", "first"),
    ).reset_index(drop=True)
    events["duration_sec"] = events.end_sec - events.start_sec
    print(f"\n{len(events)} distinct event(s):")
    display(events)
else:
    print("No anomalies survived gap-exclusion at this threshold -- try lowering AMP_THRESHOLD_ARC.")


In [ ]:
# Zoom into context (±5s) around the single largest flagged RA/PA event, across every PID/KF component.
if len(clean_flagged) and len(events):
    worst = events.loc[events.peak_amp_arcsec.idxmax()]
    print(f"Largest event: t={worst.start_sec:.2f}-{worst.end_sec:.2f}s  Az={worst.az_min:.1f}-{worst.az_max:.1f}°  "
          f"peak |error_RA|={worst.peak_amp_arcsec:.2f} arcsec  worst_corr={worst.worst_corr:.3f}")
    window = merged[(merged.t_sec > worst.start_sec - 5) & (merged.t_sec < worst.end_sec + 5)] if len(merged) else pd.DataFrame()
    display_cols = [c for c in window.columns if c.startswith(("Δ_", "α_", "θ_", "t_sec"))]
    window[display_cols]
else:
    print("No events to zoom into at this threshold.")


# Notes